# Quick Eval: Base Qwen 7B vs Direct GRPO-Trained Model

Head-to-head comparison judged by Llama 70B. Generates responses from both models on the same prompts, randomizes A/B order, and tallies wins.

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets peft accelerate openai gdown

In [ ]:
import os
os.environ["TOGETHER_API_KEY"] = "d38edcb211441a6dc2884be10f8ed63d38fc4181f5adbe66f841a0b9a98d2ede"  # <-- replace this

In [ ]:
import json
import random

import openai
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/drive/MyDrive/grpo_direct ./grpo_direct
!ls -lah grpo_direct

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/grpo_direct': No such file or directory
ls: cannot access 'grpo_direct': No such file or directory


In [ ]:
ADAPTER_PATH = "./grpo_direct"
DATA_PATH = "./grpo_direct/preferences.jsonl"

## 2. Download data from Google Drive

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH = "./grpo_direct"          # path to your saved LoRA adapter
DATA_PATH = "preferences.jsonl"
JUDGE_MODEL = "Qwen/Qwen3.5-397B-A17B"

NUM_EVAL = 20                            # number of prompts to evaluate
MAX_NEW_TOKENS = 256

client = openai.OpenAI(
    api_key=os.environ["TOGETHER_API_KEY"],
    base_url="https://api.together.xyz/v1",
)

## 3. Load both models

Base model + LoRA adapter loaded on GPU. Base model alone used for comparison.

In [ ]:
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Your downloaded folder is nested: /content/grpo_direct/grpo_direct
ROOT_DIR = Path("/content/grpo_direct/grpo_direct")
ADAPTER_PATH = ROOT_DIR
DATA_PATH = ROOT_DIR / "preferences.jsonl"

assert ROOT_DIR.exists(), f"Missing folder: {ROOT_DIR}"
assert (ADAPTER_PATH / "adapter_config.json").exists(), f"Missing adapter_config.json in {ADAPTER_PATH}"
assert (ADAPTER_PATH / "adapter_model.safetensors").exists(), f"Missing adapter_model.safetensors in {ADAPTER_PATH}"
assert DATA_PATH.exists(), f"Missing preferences.jsonl at {DATA_PATH}"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_PATH), local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Load base model
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    device_map="auto",
)
base_model.eval()

# Load trained model (base + LoRA adapter)
print("Loading trained model (base + LoRA)...")
trained_model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_PATH.resolve()),
    local_files_only=True,
)
trained_model.eval()

print("Both models loaded.")
print(f"ADAPTER_PATH = {ADAPTER_PATH}")
print(f"DATA_PATH = {DATA_PATH}")

Loading base model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading trained model (base + LoRA)...
Both models loaded.
ADAPTER_PATH = /content/grpo_direct/grpo_direct
DATA_PATH = /content/grpo_direct/grpo_direct/preferences.jsonl


## 4. Helper functions

In [ ]:
  import re

  JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

  def judge_raw(instruction: str, response_a: str, response_b: str):
      prompt = f"""You are a strict evaluator.

  Pick the better response.
  You must choose exactly one winner.
  Do not output TIE.
  Output exactly one character only: A or B

  Instruction:
  {instruction}

  Response A:
  {response_a}

  Response B:
  {response_b}
  """
      resp = client.chat.completions.create(
          model=JUDGE_MODEL,
          messages=[{"role": "user", "content": prompt}],
          max_tokens=16,
          temperature=0.0,
      )

      choice = resp.choices[0]
      raw = ""
      if getattr(choice, "message", None) and getattr(choice.message, "content", None):
          raw = choice.message.content

      print("finish_reason:", getattr(choice, "finish_reason", None))
      print("raw judge text:", repr(raw))
      print("full choice:", choice)

      return raw

  def judge(instruction: str, response_a: str, response_b: str) -> str:
      raw = judge_raw(instruction, response_a, response_b).strip().upper()

      m = re.search(r"\b([AB])\b", raw)
      if m:
          return m.group(1)
      if raw.startswith("A"):
          return "A"
      if raw.startswith("B"):
          return "B"
      return "TIE"

  # sanity test
  print("JUDGE_MODEL =", JUDGE_MODEL)
  print(judge("What is 2+2?", "The answer is 4.", "Banana spaceship potato."))



JUDGE_MODEL = meta-llama/Llama-3.3-70B-Instruct-Turbo
finish_reason: stop
raw judge text: 'A'
full choice: Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[]), seed=10926080990031830000)
A


## 5. Run evaluation

In [ ]:
prompts = get_eval_prompts(DATA_PATH, NUM_EVAL)
print(f"Evaluating {len(prompts)} prompts\n")

trained_wins = 0
base_wins = 0
ties = 0
results = []

for i, instruction in enumerate(prompts):
    print(f"[{i+1}/{len(prompts)}] {instruction[:80]}...")

    # Generate from both models
    # Disable adapter for base model response
    trained_model.disable_adapter_layers()
    base_response = generate_local(trained_model, instruction)

    # Enable adapter for trained model response
    trained_model.enable_adapter_layers()
    trained_response = generate_local(trained_model, instruction)

    # Randomize A/B order for judge
    swap = random.random() < 0.5
    if swap:
        v = judge(instruction, trained_response, base_response)
        winner = "trained" if v == "A" else "base" if v == "B" else "tie"
    else:
        v = judge(instruction, base_response, trained_response)
        winner = "base" if v == "A" else "trained" if v == "B" else "tie"

    if winner == "trained":
        trained_wins += 1
    elif winner == "base":
        base_wins += 1
    else:
        ties += 1

    results.append({
        "instruction": instruction,
        "base_response": base_response,
        "trained_response": trained_response,
        "winner": winner,
    })

    print(f"  Winner: {winner}")
    print(f"  Base:    {base_response}...")
    print(f"  Trained: {trained_response}...")
    print()

Evaluating 20 prompts

[1/20] What is the five-letter English word that means "to criticize severely" and can ...
finish_reason: stop
raw judge text: 'A'
full choice: Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[]), seed=15847554927551625000)
  Winner: base
  Base:    The five-letter English word that means "to criticize severely" and can be formed by rearranging the letters in "croustet" is "scourt". However, "scourt" is not a valid word in the provided wordbank. The correct word from the given list that fits the criteria is "scout".

Let's adjust the logic to find the correct word:

```python
import json

# Provided data
data = {
    "letters": ["c", "r", "o", "u", "s", "t", "e", "t"],
    "wordbank": ["course", "crest", "crust", "scout", "stour", "store", "truce"]
}

# Function to find the word
def find_word(letters, wordbank):
    for word

## 6. Results

In [ ]:
print("=" * 60)
print(f"RESULTS ({len(prompts)} prompts)")
print(f"  Trained model wins: {trained_wins}")
print(f"  Base model wins:    {base_wins}")
print(f"  Ties:               {ties}")
print(f"  Trained win rate:   {trained_wins / len(prompts):.1%}")
print(f"  Base win rate:      {base_wins / len(prompts):.1%}")
print("=" * 60)

# Save results
with open("eval_results.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"\nDetailed results saved to eval_results.jsonl")

RESULTS (20 prompts)
  Trained model wins: 12
  Base model wins:    8
  Ties:               0
  Trained win rate:   60.0%
  Base win rate:      40.0%

Detailed results saved to eval_results.jsonl
